# 03 - Apriori Modeling (Association Rules Mining)

Mục tiêu:
- Khai thác luật kết hợp bằng Apriori
- Phát hiện tổ hợp đặc điểm nhân viên liên quan đến Attrition
- Lọc luật theo support / confidence / lift
- Phân tích và trực quan hóa luật
- Lưu luật đã lọc để sử dụng cho các bước tiếp theo

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Thiết lập project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.mining.association import (
    discretize_features,
    build_transactions,
    mine_association_rules,
    split_rules_by_attrition
)

In [ ]:
df = pd.read_csv("../data/processed/hr_processed.csv")
print(df.shape)
df.head()

# Association Rules - Apirori

In [ ]:
# Tham số Apriori
# Tham số sinh luật
MIN_SUPPORT = 0.02

# Tham số lọc luật
FILTER_MIN_SUPPORT = 0.05
FILTER_MIN_CONF = 0.6
FILTER_MIN_LIFT = 1.2
FILTER_MAX_ANTECEDENTS = 3
FILTER_MAX_CONSEQUENTS = 1

print("Apriori parameters:")
print(f"- Min support (generate): {MIN_SUPPORT}")
print(f"- Min support (filter)  : {FILTER_MIN_SUPPORT}")
print(f"- Min confidence        : {FILTER_MIN_CONF}")
print(f"- Min lift              : {FILTER_MIN_LIFT}")

In [ ]:
df_disc = discretize_features(df)
transactions = build_transactions(df_disc)

print(f"Số transaction                 : {transactions.shape[0]:,}")
print(f"Số item trung bình / transaction: {transactions.sum(axis=1).mean():.2f}")

transactions.head()

In [ ]:
# Khai phá luật kết hợp (Apriori)
rules_ap = mine_association_rules(
    transactions,
    min_support=MIN_SUPPORT,
)

print(f"Tổng số luật sinh ra ban đầu: {rules_ap.shape[0]:,}")

rules_ap.head()

In [ ]:
# ===== LỌC, SẮP XẾP & CHỌN TOP-K LUẬT (KHÔNG ÉP CONSEQUENT) =====

n_rules_raw = rules_ap.shape[0]
print("Tổng số luật ban đầu:", n_rules_raw)

rules_filtered = rules_ap[
    (rules_ap["support"] >= FILTER_MIN_SUPPORT) &
    (rules_ap["confidence"] >= FILTER_MIN_CONF) &
    (rules_ap["lift"] >= FILTER_MIN_LIFT) &
    (rules_ap["antecedents"].apply(len) <= FILTER_MAX_ANTECEDENTS) &
    (rules_ap["consequents"].apply(len) <= FILTER_MAX_CONSEQUENTS)
].copy()

print("Sau lọc ngưỡng cơ bản:", rules_filtered.shape[0])

# ===== LOẠI LUẬT CHỨA NaN =====
def has_nan_item(items):
    return any("nan" in str(item).lower() for item in items)

rules_filtered = rules_filtered[
    ~rules_filtered["antecedents"].apply(has_nan_item) &
    ~rules_filtered["consequents"].apply(has_nan_item)
]

print("Sau loại luật chứa NaN:", rules_filtered.shape[0])

# ===== LOẠI LUẬT HIỂN NHIÊN =====
def is_trivial_rule(row):
    a = " ".join(row["antecedents"])
    c = " ".join(row["consequents"])
    return (
        ("JobLevel" in a and "MonthlyIncome" in c) or
        ("MonthlyIncome" in a and "JobLevel" in c)
    )

rules_filtered = rules_filtered[
    ~rules_filtered.apply(is_trivial_rule, axis=1)
]

print("Sau loại luật hiển nhiên:", rules_filtered.shape[0])

# ===== SORT & TOP-K =====
rules_filtered = rules_filtered.sort_values(
    by=["lift", "confidence", "support"],
    ascending=[False, False, False]
)

TOP_K = 100
rules_topk = rules_filtered.head(TOP_K)

print(f"Số luật cuối cùng (Top-{TOP_K}): {rules_topk.shape[0]}")

cols_preview = ["antecedents", "consequents", "support", "confidence", "lift"]
rules_topk[cols_preview].head(10)

In [ ]:
# =========================================================
# PHÂN TÍCH LUẬT THEO TRẠNG THÁI (ATTRITION)
# =========================================================

leave_rules_all, stay_rules_all = split_rules_by_attrition(rules_ap)

print("Tổng luật liên quan đến nghỉ việc (chưa lọc):", leave_rules_all.shape[0])
print("Tổng luật liên quan đến ở lại    (chưa lọc):", stay_rules_all.shape[0])

leave_rules = leave_rules_all[
    (leave_rules_all["support"] >= 0.01) &
    (leave_rules_all["confidence"] >= 0.3) &
    (leave_rules_all["lift"] >= 1.2)
]

stay_rules = stay_rules_all[
    (stay_rules_all["support"] >= 0.01) &
    (stay_rules_all["confidence"] >= 0.3) &
    (stay_rules_all["lift"] >= 1.2)
]

print("\nSau lọc luật Attrition:")
print("Số luật liên quan đến nghỉ việc:", leave_rules.shape[0])
print("Số luật liên quan đến ở lại    :", stay_rules.shape[0])

In [ ]:
# =========================================================
# SO SÁNH CÁC TRƯỜNG HỢP TOP_K
# =========================================================

TOP_K_LIST = [10, 20, 50, 100, 200, 300, 400]
rows = []

for k in TOP_K_LIST:
    top_leave = leave_rules.sort_values("lift", ascending=False).head(k)
    top_stay  = stay_rules.sort_values("lift", ascending=False).head(k)

    rows.append({
        "TOP_K": k,
        "Leave_Count": top_leave.shape[0],
        "Stay_Count": top_stay.shape[0],
        "Leave_Lift_Mean": top_leave["lift"].mean(),
        "Stay_Lift_Mean": top_stay["lift"].mean(),
        "Leave_Conf_Mean": top_leave["confidence"].mean(),
        "Stay_Conf_Mean": top_stay["confidence"].mean(),
    })

df_topk_compare = pd.DataFrame(rows)
display(df_topk_compare)

In [ ]:
# So sánh Lift trung bình theo TOP_K
plt.figure(figsize=(10, 6))

plt.plot(
    df_topk_compare["TOP_K"],
    df_topk_compare["Leave_Lift_Mean"],
    marker="o",
    label="Attrition = Leave"
)

plt.plot(
    df_topk_compare["TOP_K"],
    df_topk_compare["Stay_Lift_Mean"],
    marker="o",
    label="Attrition = Stay"
)

plt.xlabel("TOP_K")
plt.ylabel("Lift trung bình")
plt.title("So sánh Lift trung bình theo TOP_K")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confidence trung bình theo TOP_K
plt.figure(figsize=(10, 6))

plt.plot(
    df_topk_compare["TOP_K"],
    df_topk_compare["Leave_Conf_Mean"],
    marker="o",
    label="Leave"
)

plt.plot(
    df_topk_compare["TOP_K"],
    df_topk_compare["Stay_Conf_Mean"],
    marker="o",
    label="Stay"
)

plt.xlabel("TOP_K")
plt.ylabel("Confidence trung bình")
plt.title("So sánh Confidence trung bình theo TOP_K")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# CHỐT TOP_K CUỐI CÙNG = 100
# =========================================================

FINAL_TOP_K = 100
print("=== 100 luật tiêu biểu (theo lift) sau khi lọc ===")
final_leave_rules = (
    leave_rules
    .sort_values("lift", ascending=False)
    .head(FINAL_TOP_K)
    .copy()
)

final_stay_rules = (
    stay_rules
    .sort_values("lift", ascending=False)
    .head(FINAL_TOP_K)
    .copy()
)

# =========================================================
# TẠO TÊN LUẬT ĐỂ DÙNG CHO TRỰC QUAN
# =========================================================

def format_rule(row):
    return f"{', '.join(row['antecedents'])} → {', '.join(row['consequents'])}"

final_leave_rules["rule_name"] = final_leave_rules.apply(format_rule, axis=1)
final_stay_rules["rule_name"]  = final_stay_rules.apply(format_rule, axis=1)

# =========================================================
# HIỂN THỊ BẢNG LUẬT (CÓ TÊN LUẬT)
# =========================================================

display(
    final_leave_rules[
        ["rule_name", "support", "confidence", "lift"]
    ]
)

display(
    final_stay_rules[
        ["rule_name", "support", "confidence", "lift"]
    ]
)

# Trực quan hoá luật kết hợp

#### TOP LUẬT THEO LIFT (TOP_K = 100)

In [ ]:
# Top luật theo Lift – Leave
top_lift_leave = final_leave_rules.sort_values("lift", ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_lift_leave,
    x="lift",
    y="rule_name",
    palette="Reds_r"
)

plt.xlabel("Lift")
plt.ylabel("Luật")
plt.title("Top 10 luật theo Lift (Attrition = Leave)")
plt.tight_layout()
plt.show()

In [ ]:
# Top luật theo Lift – Stay
top_lift_stay = final_stay_rules.sort_values("lift", ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_lift_stay,
    x="lift",
    y="rule_name",
    palette="Blues_r"
)

plt.xlabel("Lift")
plt.ylabel("Luật")
plt.title("Top 10 luật theo Lift (Attrition = Stay)")
plt.tight_layout()
plt.show()

#### TOP LUẬT THEO CONFIDENCE (TOP_K = 100)

In [ ]:
# Top luật theo Confidence – Leave
top_conf_leave = final_leave_rules.sort_values("confidence", ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_conf_leave,
    x="confidence",
    y="rule_name",
    palette="Reds_r"
)

plt.xlabel("Confidence")
plt.ylabel("Luật")
plt.title("Top 10 luật theo Confidence (Attrition = Leave)")
plt.tight_layout()
plt.show()

In [ ]:
# Top luật theo Confidence – Stay
top_conf_stay = final_stay_rules.sort_values("confidence", ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_conf_stay,
    x="confidence",
    y="rule_name",
    palette="Blues_r"
)

plt.xlabel("Confidence")
plt.ylabel("Luật")
plt.title("Top 10 luật theo Confidence (Attrition = Stay)")
plt.tight_layout()
plt.show()

#### SUPPORT VS CONFIDENCE (SCATTER PLOT)

In [ ]:
# Scatter Support – Confidence – Leave
plt.figure(figsize=(8, 6))
plt.scatter(
    final_leave_rules["support"],
    final_leave_rules["confidence"],
    c=final_leave_rules["lift"],
    cmap="Reds",
    alpha=0.7
)

plt.colorbar(label="Lift")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.title("Support vs Confidence (Attrition = Leave)")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter Support – Confidence – Stay
plt.figure(figsize=(8, 6))
plt.scatter(
    final_stay_rules["support"],
    final_stay_rules["confidence"],
    c=final_stay_rules["lift"],
    cmap="Blues",
    alpha=0.7
)

plt.colorbar(label="Lift")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.title("Support vs Confidence (Attrition = Stay)")
plt.tight_layout()
plt.show()

In [ ]:
# So sánh Lift & Confidence trung bình (Leave vs Stay)
summary_topk = pd.DataFrame({
    "Group": ["Leave", "Stay"],
    "Mean_Lift": [
        final_leave_rules["lift"].mean(),
        final_stay_rules["lift"].mean()
    ],
    "Mean_Confidence": [
        final_leave_rules["confidence"].mean(),
        final_stay_rules["confidence"].mean()
    ]
})

summary_topk

In [ ]:
summary_topk.set_index("Group")[["Mean_Lift", "Mean_Confidence"]].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("So sánh Lift & Confidence trung bình (TOP_K = 100)")
plt.ylabel("Giá trị trung bình")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# LƯU LUẬT APRIORI
# =========================================================

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Toàn bộ luật Apriori (chưa lọc)
rules_ap.to_csv(
    f"{OUTPUT_DIR}/rules_apriori_all.csv",
    index=False
)

# 2. Luật đã lọc theo support / confidence / lift
rules_filtered.to_csv(
    f"{OUTPUT_DIR}/rules_apriori_filtered.csv",
    index=False
)

# 3. Top-K luật (đầu vào cho bước tiếp theo)
rules_topk.to_csv(
    f"{OUTPUT_DIR}/rules_apriori_topk.csv",
    index=False
)

# 4. Luật liên quan đến Attrition = Leave
final_leave_rules.to_csv(
    f"{OUTPUT_DIR}/rules_apriori_leave_top100.csv",
    index=False
)

# 5. Luật liên quan đến Attrition = Stay
final_stay_rules.to_csv(
    f"{OUTPUT_DIR}/rules_apriori_stay_top100.csv",
    index=False
)

print("✅ Đã lưu đầy đủ luật Apriori (TOP_K = 100):")
print("- rules_apriori_all.csv")
print("- rules_apriori_filtered.csv")
print("- rules_apriori_topk.csv")
print("- rules_apriori_leave_top100.csv")
print("- rules_apriori_stay_top100.csv")